# Listing Tables, Feature Classes, and Fields

In the ArcPy notebooks, you used list functions to find files, workspaces, feature datasets, and feature classes. Before a script can select or change attribute data, it also needs to know which tables exist, which fields each table has, and what type of data each field stores. In this notebook, you will use ArcPy list functions to answer those questions for the Austin geodatabase.

🎯 What You'll Learn

* List standalone tables in a geodatabase
* List fields and read the properties of Field objects
* Build a list of field names with a loop and with a list comprehension
* Store each feature class and its field names in a dictionary with nested loops

Work through each numbered section: a demo shows how something works, then a ✏️ **Practice** prompt below it gives you a chance to try it yourself.

### ⚙️ Before you start: select the kernel

Check that the kernel in the top-right corner of the notebook is set to `arcgispro-py3`. If it is not, click the kernel name (or **Select Kernel**) and select `arcgispro-py3`. If it is not listed, refer back to `jupyter-notebook-basics.ipynb`.

---

### 1. Setup: ArcPy, folders, and a copy of the geodatabase

This notebook will work on a copy of `Austin.gdb` instead of the original in `austin_data`. The next cell creates the `outputs\m4` folder, copies `Austin.gdb` into it, and sets the environment settings so that every step in this notebook refers to the copy. The original data in `austin_data` remains untouched.

The cell copies `Austin.gdb` only if the copy does not exist yet. Rerunning the cell keeps the copy and every change you made to it. To start over with a fresh copy, follow the commented-out lines in the cell.

⚠️ **Important**: If your working folder is not `C:\workspace\gist`, update `working_folder` below.

In [ ]:
# Import ArcPy and os
import arcpy
import os

# Store the path to your working folder. Update this path to match the
# working folder you created in course-data-setup.ipynb
working_folder = r"C:\workspace\gist"
print(f"working_folder: {working_folder}")

# Build the path to the source geodatabase. It is only read, never changed
source_gdb = os.path.join(working_folder, "austin_data", "Austin.gdb")
print(f"Austin.gdb exists: {arcpy.Exists(source_gdb)}")

# Build the path to the outputs folder for this module, and create it if
# it does not exist yet
output_folder = os.path.join(working_folder, "outputs", "m4")
os.makedirs(output_folder, exist_ok=True)

# Build the path to the copy for this notebook
gdb_path = os.path.join(output_folder, "Austin.gdb")
print(f"gdb_path: {gdb_path}")

# Copy Austin.gdb only if the copy does not exist yet, so rerunning this
# cell keeps the changes you made to the copy
if not arcpy.Exists(gdb_path):
    arcpy.management.Copy(source_gdb, gdb_path)
    print("\tCopied Austin.gdb")
else:
    print("\tThe copy already exists")

    # To start over with a fresh copy, uncomment the next three lines and
    # rerun this cell. This deletes the copy and every change you made to
    # it, then copies Austin.gdb again
    # arcpy.management.Delete(gdb_path)
    # arcpy.management.Copy(source_gdb, gdb_path)
    # print("\tDeleted the copy and copied Austin.gdb again")

# Set the workspace to the copy, and allow tools to replace outputs
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

⚠️ **Important**: If the Delete tool raises an exception saying the geodatabase is locked, another program is using the copy. Close ArcGIS Pro if it is open, then restart the kernel and rerun the cells in this section from the top.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 2. Listing tables

`arcpy.ListFeatureClasses()` finds data that has geometry, but not every table in a geodatabase has geometry. Every feature class has an **attribute table**, a table with one row per feature and one column per attribute, linked to the shapes on the map. A **standalone table** is a table with no geometry at all, such as a list of inspections or survey results. It stores records that can be related to features, but it cannot be drawn on a map by itself.

`arcpy.ListTables()` returns a list of the names of the standalone tables in the workspace. Like `arcpy.ListFiles()`, its optional first parameter is a **wildcard**, a pattern that filters names, where the asterisk `*` matches any number of characters. Wildcards in ArcPy list functions are not case sensitive.

In [ ]:
# Example 1 - List all standalone tables

# Print the current workspace set in Section 1
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# ListTables() returns a list of table names (strings)
list_of_tables = arcpy.ListTables()
print(f"list_of_tables: {list_of_tables}")
print(f"Number of tables: {len(list_of_tables)}")

# Use a for loop to print each table
print(f"Standalone tables in {arcpy.env.workspace}:")
for i, one_table in enumerate(list_of_tables, start=1):
    print(f"\t{i}. {one_table}")

In [ ]:
# Example 2 - Filter tables with a wildcard

# Matches table names that contain the letter "t" anywhere in the name.
# The wildcard is not case sensitive, so "T" also matches
t_tables = arcpy.ListTables("*t*")

# Use a for loop to print each table
print("Standalone tables with 't' in the name:")
for i, one_table in enumerate(t_tables, start=1):
    print(f"\t{i}. {one_table}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 2a] Use arcpy.ListTables() and a wildcard to store a list of all table names
# that contain the word "restaurants". Use a for loop to print the name of
# each table.

💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 3. Listing fields

A **field** is a column in a table, and each field stores the values of one attribute for every row. Every field has a **field type**, the kind of data it stores, such as text, whole numbers, decimal numbers, or dates.

`arcpy.ListFields()` differs from the other list functions in two ways:

* Its first argument is the dataset whose fields you want, either a name looked up in the workspace or a full path. It does not search the workspace for datasets.
* It does not return a list of strings. It returns a list of `Field` objects.

`Field` is an ArcPy **class**, a blueprint that defines what information an object of that kind stores. Each **object** created from a class is one instance of it, holding its own values. `arcpy.ListFields()` returns one `Field` object for each field in the dataset. Each object describes one field and stores that field's information in **properties**, named values that you read with dot notation, such as `field.name`. The properties of a `Field` object include:

* `name`: the field's name, as stored in the table
* `aliasName`: the field's **alias**, an alternate, more readable name that ArcGIS Pro can display in place of the field name
* `type`: the field type, as a string
* `length`: the **field length**, the maximum number of characters the field can store. It is mainly meaningful for String fields.

For more information about the properties of `Field` objects, see the [Field class documentation](https://doc.esri.com/en/arcgis-pro/latest/arcpy/classes/field.html).

In [ ]:
# Example 1 - ListFields() returns Field objects

# Variable stores the full path of the police_districts feature class in the
# Administrative feature dataset. We will use variable gdb_path from Section 1
police_districts_fc = os.path.join(
    gdb_path, "Administrative", "police_districts"
)
print(f"police_districts_fc: {police_districts_fc}")

# ListFields() returns a list of Field objects, not strings
pd_fields = arcpy.ListFields(police_districts_fc)
print(f"Number of fields: {len(pd_fields)}")

# Loop over the list of Field objects
for i, one_field in enumerate(pd_fields, start=1):
    # Read each property with dot notation
    print(f"\t{i}. Name: {one_field.name} | Alias: {one_field.aliasName} "
          f"| Type: {one_field.type} | Length: {one_field.length}")

The strings returned by the `type` property do not always match the labels in the **Fields** view in ArcGIS Pro. The three most common differences are:

| `type` property | ArcGIS Pro Fields view |
| --- | --- |
| `"String"` | Text |
| `"Integer"` | Long |
| `"SmallInteger"` | Short |

Python code compares against the `type` property values, so use the left column in your scripts.

`arcpy.ListFields()` has two optional parameters. The first is a wildcard that filters field names. The second, `field_type`, returns only fields of one type. It accepts the same values that the `type` property returns, such as `"String"`, `"Integer"`, `"Double"`, and `"Date"`.

In [ ]:
# Example 2 - Filter fields by name with a wildcard

# Matches field names that end with "ID"
pd_id_fields = arcpy.ListFields(police_districts_fc, "*ID")

# Print the name and type of each matching field
print(f"Fields that end with 'ID' in {police_districts_fc}:")
for i, one_field in enumerate(pd_id_fields, start=1):
    print(f"\t{i}. {one_field.name} | Type: {one_field.type}")

In [ ]:
# Example 3 - Filter fields by type

# field_type is the third parameter, so a keyword argument names it
pd_string_fields = arcpy.ListFields(police_districts_fc, field_type="String")

# Print the name and length of each String field
print(f"String fields in {police_districts_fc}:")
for i, one_field in enumerate(pd_string_fields, start=1):
    print(f"\t{i}. {one_field.name} | Length: {one_field.length}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 3a] Create a variable to store the full path of a feature class of your
# choice in Austin.gdb. Use os.path.join() to build the path.


# 3b] Use arcpy.ListFields() to store a list of Field objects for your
# feature class. Use a for loop to print the name and type of each field.


# 3c] Use arcpy.ListFields() with the field_type parameter to store a list of
# only the Double fields in your feature class. Use a for loop to print the
# name and length of each field.


# 3d] Use arcpy.ListFields() and a wildcard of your choice to store a list of
# filtered fields. Use a for loop to print the name and length of each field.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 4. Building a list of field names

The tools and cursors you will use to select and change attribute data take field names as strings, not `Field` objects. A common first step in a script is to turn the list of `Field` objects into a list of field name strings.

There are two ways to perform this task:

* A `for` loop and `.append()`, which you already know from the Lists and Loops notebooks
* A **list comprehension**, which builds a new list from another list in a single line of code. You will often see list comprehensions in Esri's documentation and in AI-generated code.

In [ ]:
# Example 1 - Build a list of field names with a for loop

# We will use police_districts_fc from Section 3
print(f"police_districts_fc: {police_districts_fc}")

# Create a list of all fields in police_districts
pd_fields = arcpy.ListFields(police_districts_fc)
# Print the field objects in the list
print(f"pd_fields: {pd_fields}")
# Print the number of field objects in the list
print(f"Number of field objects: {len(pd_fields)}")

# Start with an empty list
pd_names = []

# Loop over the Field objects and append each name to the list
for one_field in pd_fields:
    pd_names.append(one_field.name)

# Print the names in the new list
print(f"pd_names: {pd_names}")

In [ ]:
# Example 2 - Build the same list with a list comprehension

# Read it as: "one_field.name for each one_field in pd_fields"
pd_names_lc = [one_field.name for one_field in pd_fields]

# Print the names in the new list
print(f"pd_names_lc: {pd_names_lc}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 4a] Create a variable to store the full path of a feature class of your
# choice in Austin.gdb. Use os.path.join() to build the path.


# 4b] Use arcpy.ListFields() to store a list of Field objects for your
# feature class. Print the number of Field objects in your list.


# 4c] Use a for loop and .append() to build a list of the field names in
# your Field object list. Print the list.


# 4d] Use a list comprehension to build the same list of field names
# from your Field object list. Print the list.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this section, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

### 5. Storing feature classes and their fields in a dictionary

In the Loops notebook, you nested one loop inside another, and the inner loop ran completely for every pass of the outer loop. In the Dictionaries notebook, you stored values under keys and looked them up later. Listing every feature class and its fields combines both: the outer loop goes through the feature classes, the inner loop goes through the fields of the current feature class, and a dictionary stores the result. Each key is a feature class name, and each value is the list of that feature class's field names.

In [ ]:
# Example 1 - Store each feature class and its field names in a dictionary

# Print the current workspace set in Section 1
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Outer list: feature classes in the Transportation feature dataset
transportation_fcs = arcpy.ListFeatureClasses(feature_dataset="Transportation")
print(f"transportation_fcs: {transportation_fcs}")

# Start with an empty dictionary
fields_by_fc = {}

# Outer loop: one pass per feature class
for one_fc in transportation_fcs:
    # ListFields() needs the dataset, so build its full path
    fc_path = os.path.join(gdb_path, "Transportation", one_fc)

    # Start an empty list of field names for the current feature class
    fc_field_names = []

    # Inner loop: one pass per field in the current feature class
    for one_field in arcpy.ListFields(fc_path):
        fc_field_names.append(one_field.name)

    # The inner loop has finished, so the list is complete. Store it in
    # the dictionary, with the feature class name as the key
    fields_by_fc[one_fc] = fc_field_names

# Print each key and its list of field names
for fc_name, fc_field_names in fields_by_fc.items():
    print(f"\t{fc_name} ({len(fc_field_names)} fields): {fc_field_names}")

In [ ]:
# Example 2 - Build the same dictionary with a list comprehension

# Print the current workspace set in Section 1
print(f"arcpy.env.workspace: {arcpy.env.workspace}")

# Outer list: feature classes in the Transportation feature dataset
transportation_fcs = arcpy.ListFeatureClasses(feature_dataset="Transportation")
print(f"transportation_fcs: {transportation_fcs}")

# Start with an empty dictionary
fields_by_fc_lc = {}

# Outer loop: one pass per feature class
for one_fc in transportation_fcs:
    # ListFields() needs the dataset, so build its full path
    fc_path = os.path.join(gdb_path, "Transportation", one_fc)

    # The list comprehension replaces the empty list, the inner loop, and
    # .append() from Example 1. Store the list with the feature class
    # name as the key
    fields_by_fc_lc[one_fc] = [
        one_field.name for one_field in arcpy.ListFields(fc_path)
    ]

# Print each key and its list of field names
for fc_name, fc_field_names in fields_by_fc_lc.items():
    print(f"\t{fc_name} ({len(fc_field_names)} fields): {fc_field_names}")

✏️ **Practice**: Refer to the comments to write each line of code. Remember to run the cell after writing out all lines of code.

In [ ]:
# 5a] Choose a feature dataset in Austin.gdb other than Transportation.
# Use arcpy.ListFeatureClasses() to store a list of its feature classes,
# and print the list.


# 5b] Create an empty dictionary and use loops to store each feature class
# name as a key, and the list of its field names of type "String" as the
# value. Print each key and the number of fields in its list.


💾 **Important Reminder**: Stage, Commit, and Sync (Push)  
After completing this notebook, stage, commit, and sync (push) your changes, then refresh your repository page to confirm they were saved.

---

*End of notebook.*